> **Course website copy · Team 2.** This is the team's final submission as handed in, with their run outputs. Names are removed. The one cell below was added for the website: it fetches the team's data files from the course repository, so *Runtime → Run all* works without uploading anything. Everything after it is the team's own work.

In [ ]:
# Added for the course website (not part of the team's submission):
# copy the team's data files from the course repository into the folder the notebook expects.
import os, shutil, subprocess
SRC = "week07/projects/team2-bluetooth-speaker/data"
TARGET = "Bluetooth_Speaker_Dataset"
if not os.path.isdir("course_copy"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/TH-Koln-Bartnik/sscm-2026-student", "course_copy"], check=True)
    subprocess.run(["git", "-C", "course_copy", "sparse-checkout", "set", SRC], check=True)
os.makedirs(TARGET, exist_ok=True)
for name in os.listdir(os.path.join("course_copy", SRC)):
    shutil.copy(os.path.join("course_copy", SRC, name), os.path.join(TARGET, name))
print(len(os.listdir(TARGET)), "data files in", os.path.abspath(TARGET))


# RhineSound — JBL Flip 6 reference case
**Self-contained notebook + CSV dataset · three scenarios · 52 weeks · seeds 1–10**

Only this notebook and the CSV dataset are required as inputs. All parameters, simulation functions, checks and animation template are included below. Run all cells in order. In Colab upload the package ZIP when asked; locally put the notebook beside Bluetooth_Speaker_Dataset.

Generated results are saved in a Results folder. You may delete that folder and regenerate it by running this notebook. Changes to the parameters below are also used by the sensitivity checks: there is no second copy of the simulation code.

Public product specifications refer to JBL Flip 6. Assembly location, supplier compatibility and purchasing terms are hypothetical. All assumptions and provenance are documented below and in sources.csv.


In [1]:
from pathlib import Path
from zipfile import ZipFile
# Keep the notebook beside Bluetooth_Speaker_Dataset locally.
# In Colab upload this package ZIP (or the CSV files) when asked.
if not (Path.cwd()/"Bluetooth_Speaker_Dataset"/"products.csv").exists() and not Path("products.csv").exists():
    archives = list(Path.cwd().glob("*.zip"))
    if not archives:
        from google.colab import files
        uploaded = files.upload()
        archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if not Path("products.csv").exists():
        if len(archives)!=1:
            raise ValueError("Keep/upload exactly one project ZIP, or upload the dataset CSV files.")
        destination = Path("rhinesound_input").resolve()
        destination.mkdir(exist_ok=True)
        with ZipFile(archives[0]) as z:
            for name in z.namelist():
                if not (destination/name).resolve().is_relative_to(destination):
                    raise ValueError("Unsafe ZIP member")
            z.extractall(destination)
        candidates = list(destination.rglob("products.csv"))
        if len(candidates)!=1:
            raise ValueError("Expected one dataset products.csv.")
        INPUT_DATA_DIR = candidates[0].parent
if "INPUT_DATA_DIR" not in globals():
    INPUT_DATA_DIR = Path.cwd()/"Bluetooth_Speaker_Dataset" if (Path.cwd()/"Bluetooth_Speaker_Dataset").exists() else Path.cwd()
MODEL_ROOT = Path.cwd()
OUTPUT_DIR = MODEL_ROOT/"Results"
OUTPUT_DIR.mkdir(exist_ok=True)
print("CSV dataset:", INPUT_DATA_DIR.resolve())
print("Generated outputs:", OUTPUT_DIR.resolve())


CSV dataset: /workspace/scratch/b1f8fd1656f3/output/RhineSound_Minimal/Bluetooth_Speaker_Dataset
Generated outputs: /workspace/scratch/b1f8fd1656f3/output/RhineSound_Minimal/Results


## Editable assumptions

In [2]:
# All editable model assumptions are here; no parameter file needed.
CONFIG = {'demand_per_week': 300, 'order_quantity': 1000, 'setup_batch_size': 100, 'safety_z': 1.645, 'disruption_start_week_index': 20, 'disruption_duration_weeks': 3, 'backup_activation_workdays': 1, 'backup_qualified': True, 'backup_qualification_cost_eur': 300, 'holding_rate': 0.2, 'carbon_price_eur_per_tonne': 100, 'battery_energy_kwh': 0.01728, 'battery_production_kg_per_kwh': 80, 'cn_post_port_workdays': 40}


## Policy and accounting
Demand is 300 speakers/week; WC4 processing capacity is 360/week before setups, 354.10 including 10 minutes per 100 units. A 95% normal-approximation service target gives safety stock of 1,106 packs; the reorder point adds mean lead-time demand. Opening on-hand stock is safety stock plus half a purchase lot; opening pipeline is mean lead-time demand with phased residual arrivals.

The interruption blocks CN export-port access in weeks 21–23. Shipments already past the port continue. Impact reaches the plant later. The countermeasure orders 900 prequalified DE packs after one workday, with its own lead time and monthly capacity. A EUR300 annual qualification allowance is charged only to the countermeasure.

FIFO battery lots carry source-specific price, landed freight and emissions. Freight is expensed with consumption, including opening inventory; unsold inventory remains an asset. Annual on-hand holding cost uses landed inventory value. Electricity running/setup time is integrated; its purchase cost is assumed within station overhead, while its carbon cost is explicit. Non-battery kits are expensed on completed speakers; unfinished kits are not capitalized in detail.

Random processing times use gamma distributions; supplier lead times use floored normal distributions. Streams are matched by station/unit and supplier/order across scenarios. Small annual-output variation can be correct: queues can recover before year end. Shortage hours are starvation of the battery station, not simultaneous idleness at every station.


In [3]:
ANIMATION_TEMPLATE = '<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="utf-8">\n<title>Line and supply chain animation</title>\n<!-- SSCM 2026 · group project · animation template v1.0\n     Reads results.json (format: see project page). Restyle freely; the numbers must come from results.json. -->\n<style>\n  :root { --ink:#1a1a1a; --grey:#9e9e9e; --light:#f2f2f2; --red:#c1002a; --blue:#2b6cb0; --green:#2f855a; }\n  body { font-family: system-ui, -apple-system, "Segoe UI", Roboto, sans-serif; color: var(--ink); margin: 0; background: #fff; }\n  .wrap { max-width: 980px; margin: 0 auto; padding: 12px 16px 20px; }\n  h1 { font-size: 20px; margin: 4px 0 2px; }\n  .sub { color: #555; font-size: 13px; margin-bottom: 10px; }\n  .bar { display: flex; flex-wrap: wrap; gap: 8px; align-items: center; margin-bottom: 8px; }\n  button { font: inherit; font-size: 13px; padding: 6px 10px; border: 1px solid #bbb; background: #fff; border-radius: 6px; cursor: pointer; }\n  button.on { background: var(--ink); color: #fff; border-color: var(--ink); }\n  .week { font-variant-numeric: tabular-nums; font-weight: 600; min-width: 80px; }\n  input[type=range] { flex: 1; min-width: 160px; }\n  svg { width: 100%; height: auto; display: block; }\n  .tiles { display: grid; grid-template-columns: repeat(4, 1fr); gap: 8px; margin: 8px 0; }\n  .tile { background: var(--light); border-radius: 8px; padding: 8px 10px; }\n  .tile b { display: block; font-size: 22px; font-variant-numeric: tabular-nums; }\n  .tile span { font-size: 12px; color: #555; }\n  .err { background: #fdecef; border: 1px solid var(--red); padding: 10px; border-radius: 6px; white-space: pre-wrap; display: none; }\n  .loader { display: none; margin: 10px 0; }\n  .note { font-size: 11px; color: #777; margin-top: 6px; }\n</style>\n</head>\n<body>\n<div class="wrap">\n  <h1 id="title">Line and supply chain</h1>\n  <div class="sub" id="subtitle"></div>\n  <div class="loader" id="loader">Load your results file: <input type="file" id="file" accept=".json"></div>\n  <div class="err" id="err"></div>\n  <div class="bar" id="scenarioBar"></div>\n  <div class="bar"><label>Trace seed <select id="seedSelect"></select></label><span style="font-size:12px">Weekly path: selected seed. Scoreboard: mean of all ten seeds. Compare all seeds to see variability.</span></div>\n  <div class="bar">\n    <button id="play">▶ Play</button>\n    <span class="week" id="weekLabel">Week 1</span>\n    <input type="range" id="slider" min="0" max="51" value="0">\n    <label style="font-size:13px">speed\n      <select id="speed"><option value="1">slow</option><option value="2" selected>normal</option><option value="4">fast</option></select>\n    </label>\n  </div>\n  <svg id="scene" viewBox="0 60 960 270" role="img" aria-label="line and supply chain"></svg>\n  <div class="tiles" id="tiles"></div>\n  <svg id="chart" viewBox="0 0 960 170" role="img" aria-label="units finished per week"></svg>\n  <div class="note" id="note"></div>\n</div>\n<script>\nconst RESULTS = /*RESULTS_JSON*/null;\nconst ORDER = ["baseline", "disruption", "countermeasure"];\nconst COLOURS = { baseline: "#555", disruption: "#c1002a", countermeasure: "#2b6cb0" };\nconst NS = "http://www.w3.org/2000/svg";\nlet R, scen = "baseline", week = 0, playing = false, phase = 0, last = 0;\n\nfunction el(tag, attrs, parent, text) {\n  const e = document.createElementNS(NS, tag);\n  for (const k in attrs) e.setAttribute(k, attrs[k]);\n  if (text !== undefined) e.textContent = text;\n  if (parent) parent.appendChild(e);\n  return e;\n}\nfunction fmt(x, d = 0) { return Number(x).toLocaleString("en-US", { minimumFractionDigits: d, maximumFractionDigits: d }); }\n\nfunction validate(r) {\n  const errs = [];\n  if (!r || typeof r !== "object") return ["results is empty"];\n  ["product", "unit", "stations", "scenarios"].forEach(k => { if (!(k in r)) errs.push(`missing key "${k}"`); });\n  if (errs.length) return errs;\n  ORDER.forEach(s => {\n    const sc = r.scenarios[s];\n    if (!sc) { errs.push(`scenarios lacks "${s}"`); return; }\n    ["units_per_week", "cost_per_unit_eur", "kg_co2e_per_unit", "hours_stopped"].forEach(k => {\n      if (!sc.scoreboard || typeof sc.scoreboard[k] !== "number") errs.push(`${s}.scoreboard.${k} missing or not a number`);\n    });\n    const w = sc.weeks || {};\n    ["queue", "in_transit", "stock", "finished", "stopped_hours"].forEach(k => {\n      if (!Array.isArray(w[k])) errs.push(`${s}.weeks.${k} missing (one value per week)`);\n    });\n    if (Array.isArray(w.queue) && w.queue.some(q => !Array.isArray(q) || q.length !== r.stations.length))\n      errs.push(`${s}.weeks.queue: every week needs one number per station (${r.stations.length})`);\n    const n = Array.isArray(w.finished) ? w.finished.length : 0;\n    ["queue", "in_transit", "stock", "stopped_hours"].forEach(k => {\n      if (Array.isArray(w[k]) && w[k].length !== n) errs.push(`${s}.weeks.${k} has ${w[k].length} weeks, finished has ${n}`);\n    });\n  });\n  return errs;\n}\n\nfunction start(r) {\n  const errs = validate(r);\n  const box = document.getElementById("err");\n  if (errs.length) { box.style.display = "block"; box.textContent = "results.json is not complete yet:\\n• " + errs.join("\\n• "); return; }\n  box.style.display = "none";\n  R = r;\n  const seedBox=document.getElementById(\'seedSelect\');\n  if(!seedBox.options.length) {\n    Object.keys(r.scenarios.baseline.seed_paths || {\'1\':null}).forEach(k=>seedBox.add(new Option(k,k)));\n    seedBox.onchange=()=>{ORDER.forEach(s=>{R.scenarios[s].weeks=R.scenarios[s].seed_paths[seedBox.value]});start(R);};\n  }\n  document.getElementById("title").textContent = `${r.product} · line and supply chain`;\n  document.getElementById("subtitle").textContent =\n    `Key component: ${r.key_component || "key component"} from ${r.supplier || "the supplier"}` +\n    (r.transport ? ` by ${r.transport}` : "") + ` · one year, week by week`;\n  const nW = r.scenarios.baseline.weeks.finished.length;\n  document.getElementById("slider").max = nW - 1;\n  const bar = document.getElementById("scenarioBar");\n  bar.innerHTML = "";\n  ORDER.forEach(s => {\n    const b = document.createElement("button");\n    b.textContent = r.scenarios[s].label || s;\n    b.onclick = () => { scen = s; draw(); };\n    b.dataset.s = s;\n    bar.appendChild(b);\n  });\n  // one dot stands for this many units in the queues (same scale in every scenario)\n  let qmax = 1;\n  ORDER.forEach(s => r.scenarios[s].weeks.queue.forEach(q => q.forEach(v => qmax = Math.max(qmax, v))));\n  const raw = qmax / 10, mag = Math.pow(10, Math.floor(Math.log10(raw)));\n  R._dot = Math.max(1, [1, 2, 5, 10].map(m => m * mag).find(v => v >= raw) || 10 * mag);\n  let smax = 1;\n  ORDER.forEach(s => r.scenarios[s].weeks.stock.forEach(v => smax = Math.max(smax, v)));\n  R._smax = smax;\n  document.getElementById("note").textContent =\n    `Week-end queues and stock; trace seed ${seedBox.value}. One dot = ${fmt(R._dot)} waiting units. Tiles are ten-seed means, not this trace. Moving symbols are illustrative; numerical counts come from results.json. Port access is blocked in weeks 21–23; downstream arrivals can be affected later. Partial footprint: battery + inbound transport + factory electricity.`;\n  draw();\n}\n\nfunction draw() {\n  const r = R, sc = r.scenarios[scen], w = sc.weeks, n = r.stations.length;\n  document.querySelectorAll("#scenarioBar button").forEach(b => b.classList.toggle("on", b.dataset.s === scen));\n  document.getElementById("slider").value = week;\n  document.getElementById("weekLabel").textContent = `Week ${week + 1}`;\n  const svg = document.getElementById("scene");\n  svg.innerHTML = "";\n  const down = (w.supplier_down || [])[week] === true;\n  const stopped = w.stopped_hours[week] > 0;\n\n  // supplier\n  el("rect", { x: 10, y: 120, width: 130, height: 70, rx: 8, fill: down ? "#fdecef" : "#f2f2f2", stroke: down ? "#c1002a" : "#9e9e9e", "stroke-width": 2 }, svg);\n  el("text", { x: 75, y: 145, "text-anchor": "middle", "font-size": 13, "font-weight": 600 }, svg, "Supplier");\n  el("text", { x: 75, y: 163, "text-anchor": "middle", "font-size": 11 }, svg, (r.supplier || "").slice(0, 22));\n  el("text", { x: 75, y: 180, "text-anchor": "middle", "font-size": 11, fill: down ? "#c1002a" : "#2f855a", "font-weight": 600 }, svg, down ? "CN PORT BLOCKED" : "CN port available");\n\n  // transport lane with moving dots\n  el("line", { x1: 145, y1: 155, x2: 245, y2: 155, stroke: "#9e9e9e", "stroke-width": 3, "stroke-dasharray": "6 5" }, svg);\n  el("text", { x: 195, y: 140, "text-anchor": "middle", "font-size": 11 }, svg, `in transit: ${fmt(w.in_transit[week])}`);\n  const trucks = Math.min(4, Math.ceil(w.in_transit[week] / Math.max(1, R._smax / 4)));\n  for (let i = 0; i < trucks; i++) {\n    const x = 150 + ((phase + i / Math.max(1, trucks)) % 1) * 90;\n    el("rect", { x: x, y: 147, width: 14, height: 16, rx: 2, fill: "#555" }, svg);\n  }\n\n  // stock of the key component\n  const sh = 120 * Math.min(1, w.stock[week] / R._smax);\n  el("rect", { x: 250, y: 95, width: 70, height: 120, fill: "#fff", stroke: "#9e9e9e", "stroke-width": 2 }, svg);\n  el("rect", { x: 252, y: 213 - sh, width: 66, height: sh, fill: w.stock[week] > 0 ? "#2b6cb0" : "#c1002a", opacity: 0.8 }, svg);\n  el("text", { x: 285, y: 85, "text-anchor": "middle", "font-size": 12, "font-weight": 600 }, svg, "Stock");\n  el("text", { x: 285, y: 232, "text-anchor": "middle", "font-size": 11 }, svg, `${fmt(w.stock[week])} ${r.key_component ? "parts" : ""}`);\n\n  // stations\n  const x0 = 345, x1 = 860, step = (x1 - x0) / n, bw = Math.min(90, step - 14);\n  const q = w.queue[week];\n  const worst = q.indexOf(Math.max(...q));\n  el("line", { x1: 322, y1: 155, x2: x1 + 10, y2: 155, stroke: "#ddd", "stroke-width": 6 }, svg);\n  r.stations.forEach((name, i) => {\n    const cx = x0 + step * i + step / 2, bx = cx - bw / 2;\n    const hot = q[i] > 0 && i === worst;\n    const starved = i===2 && w.stock[week]===0 && q[i]>0;\n    const isBusy = (w.busy || [])[week]?.[i]>0;\n    el("rect", { x: bx, y: 130, width: bw, height: 50, rx: 6, fill: starved ? "#fdecef" : (hot ? "#c1002a" : isBusy ? "#2b6cb0" : "#9e9e9e"), stroke: starved ? "#c1002a" : "none", "stroke-width": 2 }, svg);\n    const words = String(name).split(/[\\s/]+/);\n    const lines = [words.slice(0, Math.ceil(words.length / 2)).join(" "), words.slice(Math.ceil(words.length / 2)).join(" ")].filter(Boolean);\n    lines.forEach((t, k) => el("text", { x: cx, y: 197 + k * 13, "text-anchor": "middle", "font-size": 10.5 }, svg, t.slice(0, 18)));\n    el("text", { x: cx, y: 160, "text-anchor": "middle", "font-size": 11, fill: starved ? "#c1002a" : "#fff", "font-weight": 600 }, svg, starved ? "no battery" : `S${i + 1} ${isBusy ? \'busy\' : \'idle\'}`);\n    // queue in front of the station: stacked dots\n    const dots = Math.min(20, Math.ceil(q[i] / R._dot - 1e-9));\n    for (let d = 0; d < dots; d++) {\n      el("circle", { cx: cx - bw / 2 + 8 + (d % 5) * 9, cy: 118 - Math.floor(d / 5) * 9, r: 3.6, fill: hot ? "#c1002a" : "#1a1a1a" }, svg);\n    }\n    el("text", { x: cx, y: 118 - Math.ceil(Math.max(dots, 1) / 5) * 9 - 4, "text-anchor": "middle", "font-size": 10, fill: "#555" }, svg, `${fmt(q[i])} waiting`);\n  });\n  // a part moving along the line\n  if (!stopped) el("circle", { cx: x0 + phase * (x1 - x0), cy: 155, r: 4, fill: "#2f855a" }, svg);\n\n  // finished goods\n  el("text", { x: 930, y: 140, "text-anchor": "end", "font-size": 12, "font-weight": 600 }, svg, "Finished");\n  el("text", { x: 930, y: 162, "text-anchor": "end", "font-size": 20, "font-weight": 700 }, svg, fmt(w.finished[week]));\n  el("text", { x: 930, y: 178, "text-anchor": "end", "font-size": 11 }, svg, `${r.unit} this week`);\n  const cum = w.finished.slice(0, week + 1).reduce((a, b) => a + b, 0);\n  el("text", { x: 930, y: 196, "text-anchor": "end", "font-size": 11, fill: "#555" }, svg, `${fmt(cum)} so far`);\n  if (stopped) el("text", { x: 600, y: 290, "text-anchor": "middle", "font-size": 14, fill: "#c1002a", "font-weight": 700 }, svg,\n    `Battery station starved for ${fmt(w.stopped_hours[week], 1)} working hours this week`);\n  el("text", { x: 10, y: 315, "font-size": 11, fill: "#777" }, svg, `Scenario: ${sc.label || scen}`);\n\n  // scoreboard tiles\n  const b = sc.scoreboard;\n  document.getElementById("tiles").innerHTML = [\n    [fmt(b.units_per_week, 1), `${r.unit} per week`],\n    [fmt(b.cost_per_unit_eur, 2) + " €", "total cost per unit (one year)"],\n    [fmt(b.kg_co2e_per_unit, 2), "kg CO₂e per unit"],\n    [fmt(b.hours_stopped, 1) + " h", "line stood still (year)"],\n  ].map(([v, l]) => `<div class="tile"><b>${v}</b><span>${l}</span></div>`).join("");\n\n  drawChart();\n}\n\nfunction drawChart() {\n  const svg = document.getElementById("chart");\n  svg.innerHTML = "";\n  const L = 50, Rr = 940, T = 15, B = 140, nW = R.scenarios.baseline.weeks.finished.length;\n  let ymax = 1;\n  ORDER.forEach(s => R.scenarios[s].weeks.finished.forEach(v => ymax = Math.max(ymax, v)));\n  ymax *= 1.1;\n  const X = i => L + (Rr - L) * i / Math.max(1, nW - 1), Y = v => B - (B - T) * v / ymax;\n  el("line", { x1: L, y1: B, x2: Rr, y2: B, stroke: "#999" }, svg);\n  el("text", { x: L, y: 10, "font-size": 11, fill: "#555" }, svg, `${R.unit} finished per week`);\n  [0, Math.round(ymax / 2), Math.round(ymax)].forEach(v => el("text", { x: L - 6, y: Y(v) + 4, "text-anchor": "end", "font-size": 10, fill: "#777" }, svg, fmt(v)));\n  ORDER.forEach(s => {\n    const pts = R.scenarios[s].weeks.finished.map((v, i) => `${X(i)},${Y(v)}`).join(" ");\n    el("polyline", { points: pts, fill: "none", stroke: COLOURS[s], "stroke-width": s === scen ? 2.8 : 1.2, opacity: s === scen ? 1 : 0.45 }, svg);\n  });\n  el("line", { x1: X(week), y1: T, x2: X(week), y2: B, stroke: "#1a1a1a", "stroke-dasharray": "3 3" }, svg);\n  ORDER.forEach((s, k) => {\n    el("rect", { x: L + k * 250, y: 152, width: 12, height: 4, fill: COLOURS[s] }, svg);\n    el("text", { x: L + k * 250 + 18, y: 158, "font-size": 11 }, svg, (R.scenarios[s].label || s).slice(0, 40));\n  });\n}\n\nfunction tick(t) {\n  if (playing && R) {\n    const speed = +document.getElementById("speed").value;\n    const dt = last ? (t - last) / 1000 : 0;\n    phase += dt * speed * 0.8;\n    if (phase >= 1) {\n      phase = 0;\n      week += 1;\n      if (week >= R.scenarios[scen].weeks.finished.length) { week = R.scenarios[scen].weeks.finished.length - 1; playing = false; document.getElementById("play").textContent = "▶ Play"; }\n    }\n    draw();\n  }\n  last = t;\n  requestAnimationFrame(tick);\n}\n\ndocument.getElementById("play").onclick = () => {\n  if (!R) return;\n  if (!playing && week >= R.scenarios[scen].weeks.finished.length - 1) week = 0;\n  playing = !playing;\n  document.getElementById("play").textContent = playing ? "❚❚ Pause" : "▶ Play";\n};\ndocument.getElementById("slider").oninput = e => { week = +e.target.value; phase = 0; if (R) draw(); };\ndocument.getElementById("file").onchange = e => {\n  const f = e.target.files[0];\n  if (!f) return;\n  f.text().then(t => { try { start(JSON.parse(t)); } catch (x) { const b = document.getElementById("err"); b.style.display = "block"; b.textContent = "Not a valid JSON file: " + x.message; } });\n};\nif (RESULTS && typeof RESULTS === "object") start(RESULTS);\nelse document.getElementById("loader").style.display = "block";\nrequestAnimationFrame(tick);\n</script>\n</body>\n</html>\n'


In [4]:
"""RhineSound / JBL Flip 6 teaching model. Run: python simulation.py.

Five-station discrete-event line; FIFO battery lots; supplier capacity ledger;
three paired scenarios. All time is scheduled working hours (40 h/week).
"""
from pathlib import Path
from collections import deque, defaultdict
import csv, json, math, heapq, itertools
from statistics import mean
import numpy as np

ROOT = Path(globals().get('MODEL_ROOT', Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()))
DATA_DIR = str(INPUT_DATA_DIR.resolve())

def read_csv(name):
    with open(Path(DATA_DIR) / name, encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

PRODUCT = read_csv('products.csv')[0]
BOM = read_csv('bom.csv')
WORKCENTERS = sorted(read_csv('workcenters.csv'), key=lambda x:int(x['seq']))
ROUTING = sorted(read_csv('routing.csv'), key=lambda x:int(x['seq']))
ENV = {r['workcenter_id']:r for r in read_csv('process_environment.csv')}
SUPPLIERS = {r['supplier_id']:r for r in read_csv('suppliers.csv')}
LANES = read_csv('network_lanes.csv')
RATES = {r['lane_id']:r for r in read_csv('freight_rates.csv')}
MODES = {r['mode_id']:r for r in read_csv('transport_emission_factors.csv')}
LOCATIONS = read_csv('locations.csv')
KEY_PART = next(r for r in BOM if r['supplier_id'])
KEY_STATION = next(i for i,w in enumerate(WORKCENTERS) if w['workcenter_id']==KEY_PART['added_at_workcenter'])
GRID = next(float(r['value']) for r in read_csv('emission_factors.csv') if r['scope']=='electricity')
# CONFIG is defined in the parameter cell above.
assert float(KEY_PART['qty_per_unit'])==1, 'This model consumes one battery per speaker.'
assert all(float(w['shift_hours'])*float(w['days_per_week'])==40 for w in WORKCENTERS)
WEEKS, HOURS_WEEK, SEEDS = 52, 40., list(range(1,11))
HORIZON = WEEKS * HOURS_WEEK

def supplier_path(sid):
    plant = next(r['location_id'] for r in LOCATIONS if r['type']=='plant')
    queue = deque([(SUPPLIERS[sid]['location_id'], [])]); seen=set()
    while queue:
        node, path = queue.popleft()
        if node == plant:return path
        if node in seen:continue
        seen.add(node)
        queue.extend((r['destination_id'],path+[r]) for r in LANES if r['origin_id']==node)
    raise ValueError('No supplier route: '+sid)

def freight(qty, sid):
    cost = co2 = 0.
    for lane in supplier_path(sid):
        rate=RATES[lane['lane_id']]
        tkm=qty*float(KEY_PART['mass_kg'])/1000*float(lane['distance_km'])
        cost+=max(float(rate['min_charge_eur']),float(rate['rate_eur_per_shipment'])+tkm*float(rate['rate_eur_per_tkm']))
        co2+=tkm*float(MODES[lane['mode_id']]['g_co2e_per_tkm'])/1000
    return cost,co2

def run_scenario(seed, scenario='baseline', overrides=None):
    p=dict(CONFIG);p.update(overrides or {})
    assert scenario in ('baseline','disruption','countermeasure')
    demand=int(p['demand_per_week']);q=int(p['order_quantity'])
    cn=SUPPLIERS['SUP-CN'];de=SUPPLIERS['SUP-DE']
    lead_weeks=float(cn['lead_time_mean_days'])*p.get('cn_lead_multiplier',1)/5
    safety=math.ceil(p['safety_z']*demand*float(cn['lead_time_sd_days'])/5)
    rop=math.ceil(demand*lead_weeks+safety)
    initial_stock=safety+q//2
    initial_pipeline=round(demand*lead_weeks)
    start=p['disruption_start_week_index']*HOURS_WEEK
    end=start+p['disruption_duration_weeks']*HOURS_WEEK
    n=len(WORKCENTERS);max_units=WEEKS*demand
    # Index by station and unit, and by supplier and order number: paired random
    # numbers stay aligned even if the event ordering changes between scenarios.
    process=[]
    for i,(wc,rt) in enumerate(zip(WORKCENTERS,ROUTING)):
        mu=float(rt['process_time_min'])*(p.get('wc4_time_multiplier',1) if i==3 else 1)
        cv=float(wc['process_time_cv'])
        rng=np.random.default_rng(np.random.SeedSequence([seed,100,i]))
        process.append(rng.gamma(1/cv**2,mu*cv**2,max_units) if cv else np.full(max_units,mu))
    lead_rng={sid:np.random.default_rng(np.random.SeedSequence([seed,200,j])) for j,sid in enumerate(SUPPLIERS)}
    queues=[deque() for _ in range(n)];busy=[0]*n;assigned=[0]*n
    active=[[] for _ in range(n)]; completed_station=[0]*n
    heap=[];serial=itertools.count();lots=deque()
    orders=[];capacity=defaultdict(int);pending={};order_index=0
    stock=0;used=defaultdict(int);received=defaultdict(int)
    consumed_price=consumed_freight=consumed_co2=0.
    stock_area=stock_value_area=0.;last_stock=0.
    stop_start=None;stop_intervals=[]
    starts=[];finishes=[];weekly_finished=[0]*WEEKS
    weekly_started=[0]*WEEKS;snapshots=[]
    initial_freight=freight(q,'SUP-CN')[0]/q

    def emit(t,priority,kind,payload=None):
        if t<=HORIZON+1e-8:heapq.heappush(heap,(float(t),priority,next(serial),kind,payload))

    def stock_integral(t):
        nonlocal stock_area,stock_value_area,last_stock
        stock_area+=stock*(t-last_stock)
        stock_value_area+=sum(x['qty']*(x['price']+x['freight']) for x in lots)*(t-last_stock)
        last_stock=t

    def add_lot(t,qty,sid,freight_unit):
        nonlocal stock
        stock_integral(t)
        lots.append(dict(qty=qty,sid=sid,price=float(SUPPLIERS[sid]['unit_price_eur'])*p.get('battery_price_multiplier',1),
                         freight=freight_unit,co2=float(SUPPLIERS[sid]['kg_co2e_per_unit'])+
                         p['battery_energy_kwh']*(p.get('production_intensity',p['battery_production_kg_per_kwh'])-p['battery_production_kg_per_kwh'])))
        stock+=qty

    add_lot(0,initial_stock,'SUP-CN',initial_freight)

    def schedule_order(t,qty,sid,initial_due=None):
        nonlocal order_index
        supplier=SUPPLIERS[sid]
        # A planning month is exactly 20 working days = 160 working hours.
        # Aggregate every accepted order, including the emergency order.
        month=int(t//160);cap=int(supplier['capacity_per_month'])
        if qty>cap:raise ValueError('Order exceeds monthly capacity')
        if initial_due is None:
            assert qty>=int(supplier['min_per_month'])
            while capacity[sid,month]+qty>cap:month+=1
            capacity[sid,month]+=qty
            accepted=max(t,month*160.)
            mu=float(supplier['lead_time_mean_days'])*(p.get('cn_lead_multiplier',1) if sid=='SUP-CN' else 1)
            # CN draw cannot be shorter than the separately tracked downstream leg.
            minimum=p['cn_post_port_workdays']+1 if sid=='SUP-CN' else 1.
            days=max(minimum,lead_rng[sid].normal(mu,float(supplier['lead_time_sd_days'])))
            due=accepted+days*8
        else:
            # Pre-year orders are already committed; evenly phased residual lead.
            accepted=-1.;due=initial_due
        planned_due=due
        # Port access failure: delay departures scheduled within [start,end).
        # Cargo already past the port is not delayed; downstream travel is retained.
        remaining=p['cn_post_port_workdays']*8
        depart=due-remaining if sid=='SUP-CN' else due-8
        delay=0.
        if scenario!='baseline' and sid=='SUP-CN' and start<=depart<end:
            delay=end-depart;due+=delay
        oid=order_index;order_index+=1
        item=dict(id=oid,supplier=sid,qty=qty,ordered_at=t,accepted_at=accepted,
                  planned_arrival=planned_due,arrival=due,port_departure=depart+delay,
                  delay_hours=delay,initial=initial_due is not None,
                  capacity_month=None if initial_due is not None else month)
        orders.append(item);pending[oid]=item;emit(due,0,'delivery',oid)

    # Initial pipeline equals expected demand during lead time; no empty-pipeline
    # startup artifact. Residual deliveries spread over one mean lead time.
    remaining=initial_pipeline;cum=0
    while remaining:
        qty=min(q,remaining);cum+=qty
        due=(cum-qty/2)/demand*HOURS_WEEK
        schedule_order(0,qty,'SUP-CN',initial_due=due)
        remaining-=qty

    def replenish(t):
        while stock+sum(x['qty'] for x in pending.values())<=rop:
            schedule_order(t,q,'SUP-CN')

    def start_work(i,t):
        nonlocal stock,stop_start,consumed_price,consumed_freight,consumed_co2
        wc=WORKCENTERS[i];rt=ROUTING[i]
        while busy[i]<int(wc['num_servers']) and queues[i]:
            if i==KEY_STATION and stock<1:
                if stop_start is None:stop_start=t
                break
            if i==KEY_STATION and stop_start is not None:
                stop_intervals.append((stop_start,t));stop_start=None
            uid=queues[i].popleft()
            if i==KEY_STATION:
                stock_integral(t);lot=lots[0];lot['qty']-=1;stock-=1
                used[lot['sid']]+=1;consumed_price+=lot['price'];consumed_freight+=lot['freight'];consumed_co2+=lot['co2']
                if lot['qty']==0:lots.popleft()
                replenish(t)
            setup=float(rt['setup_time_min']) if assigned[i]%p['setup_batch_size']==0 else 0.
            minutes=process[i][uid]+setup;assigned[i]+=1
            duration=minutes/(60*float(wc['availability']))
            # Busy interval integrates active running+setup electricity, with
            # availability losses treated as idle and final partial jobs clipped.
            active[i].append((t,min(HORIZON,t+duration),float(wc['availability'])))
            busy[i]+=1;emit(t+duration,1,'done',(i,uid))

    for uid in range(max_units):emit(uid*HOURS_WEEK/demand,2,'release',uid)
    for week in range(WEEKS):emit((week+1)*HOURS_WEEK,1.5,'snapshot',week)
    if scenario=='countermeasure':emit(start+p['backup_activation_workdays']*8,0,'backup')
    replenish(0)
    while heap:
        t,_,_,kind,payload=heapq.heappop(heap)
        if kind=='release':
            starts.append(t);finishes.append(math.inf)
            weekly_started[min(WEEKS-1,int(t//HOURS_WEEK))]+=1
            queues[0].append(payload);start_work(0,t)
        elif kind=='done':
            i,uid=payload;busy[i]-=1;completed_station[i]+=1;start_work(i,t)
            if i==n-1:
                finishes[uid]=t;weekly_finished[min(WEEKS-1,int(t//HOURS_WEEK))]+=1
            else:queues[i+1].append(uid);start_work(i+1,t)
        elif kind=='delivery':
            item=pending.pop(payload);qty=item['qty'];sid=item['supplier']
            # Initial lots retain pre-year purchase-lot freight, including small
            # residual initial deliveries; no artificial small-shipment surcharge.
            fu=initial_freight if item['initial'] else freight(qty,sid)[0]/qty
            add_lot(t,qty,sid,fu);received[sid]+=qty;start_work(KEY_STATION,t)
        elif kind=='backup':
            if p['backup_qualified']:
                qty=math.ceil(demand*p['disruption_duration_weeks'])
                schedule_order(t,qty,'SUP-DE')
        elif kind=='snapshot':
            # Exclude any release at the next week's boundary from week-end queues
            # by taking snapshots immediately before releases (priority patched below).
            snapshots.append(dict(week=payload+1,queue=[len(x) for x in queues],busy=list(busy),
                stock=stock,on_order=sum(x['qty'] for x in pending.values()),
                in_transit=sum(x['qty'] for x in pending.values() if x['port_departure']<=t),
                supplier_down=(scenario!='baseline' and start<=payload*HOURS_WEEK<end)))
    stock_integral(HORIZON)
    if stop_start is not None:stop_intervals.append((stop_start,HORIZON))
    weekly_stopped=[sum(max(0,min(b,(w+1)*40)-max(a,w*40)) for a,b in stop_intervals) for w in range(WEEKS)]
    a=np.array(starts);b=np.array(finishes);done=np.isfinite(b)
    finished=int(done.sum());started=len(a)
    if finished==0:raise ValueError('No completed speakers')
    wip=float((np.minimum(b,HORIZON)-a).sum()/HORIZON)
    flow=float((b[done]-a[done]).mean())
    electric=0.;processing=[]
    for i,wc in enumerate(WORKCENTERS):
        env=ENV[wc['workcenter_id']];servers=int(wc['num_servers'])
        run=sum((end-begin)*av for begin,end,av in active[i]);processing.append(run)
        electric+=run*float(env['kw_running'])+(HORIZON*servers-run)*float(env['kw_idle'])
    co2=consumed_co2+electric*GRID
    nonkey=sum(float(r['qty_per_unit'])*float(r['unit_cost_eur']) for r in BOM if r['component_id']!=KEY_PART['component_id'])
    materials=finished*nonkey+consumed_price
    fixed=WEEKS*sum(float(w['fixed_cost_eur_week'])*int(w['num_servers']) for w in WORKCENTERS)
    holding=p['holding_rate']*stock_value_area/HORIZON
    carbon=p['carbon_price_eur_per_tonne']*co2/1000
    activation=p['backup_qualification_cost_eur'] if scenario=='countermeasure' and p['backup_qualified'] else 0.
    total=materials+fixed+consumed_freight+holding+carbon+activation
    ending_value=sum(x['qty']*(x['price']+x['freight']) for x in lots)
    return dict(scenario=scenario,seed=seed,weeks_simulated=WEEKS,hours_simulated=HORIZON,
        units_started=started,units_finished=finished,key_components_used=sum(used.values()),
        avg_units_on_line=wip,avg_hours_on_line=flow,cost_per_unit_eur=total/finished,
        units_per_week=finished/WEEKS,kg_co2e_per_unit=co2/finished,hours_stopped=sum(weekly_stopped),
        initial_stock=initial_stock,initial_pipeline=initial_pipeline,reorder_point=rop,safety_stock=safety,
        avg_battery_stock=stock_area/HORIZON,end_battery_stock=stock,
        units_used_by_supplier=dict(used),units_received_by_supplier=dict(received),
        materials_eur=materials,fixed_cost_eur=fixed,transport_eur=consumed_freight,
        holding_eur=holding,carbon_eur=carbon,backup_qualification_eur=activation,total_cost_eur=total,
        electricity_kwh=electric,production_and_inbound_co2_kg=consumed_co2,factory_co2_kg=electric*GRID,
        ending_inventory_value_eur=ending_value,
        weekly_started=weekly_started,weekly_finished=weekly_finished,weekly_stopped=weekly_stopped,
        snapshots=snapshots,orders=orders,capacity_ledger=[dict(supplier=s,month=m,qty=v) for (s,m),v in capacity.items()],
        actual_processing_hours_by_station=processing,completed_at_station=completed_station,
        finite_horizon_little_gap=abs(wip-finished/HORIZON*flow)/(finished/HORIZON*flow))

SCORE_KEYS=['units_per_week','cost_per_unit_eur','kg_co2e_per_unit','hours_stopped']

def weekly_path(r):
    ss=r['snapshots']
    return dict(queue=[x['queue'] for x in ss],in_transit=[x['in_transit'] for x in ss],stock=[x['stock'] for x in ss],
        finished=r['weekly_finished'],stopped_hours=r['weekly_stopped'],supplier_down=[x['supplier_down'] for x in ss],
        on_order=[x['on_order'] for x in ss],busy=[x['busy'] for x in ss])

def validate_runs(runs):
    for r in runs:
        assert r['units_finished']<=r['key_components_used']<=r['units_started']
        assert r['initial_stock']+sum(r['units_received_by_supplier'].values())-r['key_components_used']==r['end_battery_stock']
        assert sum(r['weekly_finished'])==r['units_finished']
        assert math.isclose(sum(r['weekly_stopped']),r['hours_stopped'])
        assert len(r['snapshots'])==WEEKS
        assert math.isclose(sum(r[k] for k in ['materials_eur','fixed_cost_eur','transport_eur','holding_eur','carbon_eur','backup_qualification_eur']),r['total_cost_eur'])
        for c in r['capacity_ledger']:assert c['qty']<=int(SUPPLIERS[c['supplier']]['capacity_per_month'])
        for o in r['orders']:
            assert o['arrival']>=o['planned_arrival']
            if o['delay_hours']>0:assert o['supplier']=='SUP-CN'
        assert all(x['stock']>=0 and all(v>=0 for v in x['queue']) for x in r['snapshots'])
    return True

def run_all(output_dir=OUTPUT_DIR):
    output_dir=Path(output_dir); output_dir.mkdir(exist_ok=True,parents=True)
    scenarios={s:[run_scenario(seed,s) for seed in SEEDS] for s in ['baseline','disruption','countermeasure']}
    validate_runs([r for runs in scenarios.values() for r in runs])
    results=dict(product=PRODUCT['product_name'],unit='speakers',key_component=KEY_PART['component_name'],
        supplier='Model CN supplier / qualified DE backup',transport='CN sea via Yantian; DE truck',
        stations=[w['workcenter_name'] for w in WORKCENTERS],
        metadata=dict(representative_seed=1,queue_and_stock='week-end after completions; before next-week releases',
                      scoreboard='means of seeds 1-10; min/max in ranges',clock='scheduled working hours; 40/week',
                      co2_scope='battery production proxy + inbound transport + factory electricity; not whole-product LCA'),scenarios={})
    rows=[]
    for s,runs in scenarios.items():
        r=runs[0];snaps=r['snapshots'];score={k:mean(x[k] for x in runs) for k in SCORE_KEYS}
        ranges={k:dict(min=min(x[k] for x in runs),max=max(x[k] for x in runs)) for k in SCORE_KEYS}
        results['scenarios'][s]=dict(label=s.title(),scoreboard=score,ranges=ranges,weeks=weekly_path(r),
            seed_paths={str(x['seed']):weekly_path(x) for x in runs})
        for k in SCORE_KEYS:rows.append(dict(scenario=s,metric=k,mean=score[k],minimum=ranges[k]['min'],maximum=ranges[k]['max']))
        print(s, {k:round(v,4) for k,v in score.items()})
    (output_dir/'results.json').write_text(json.dumps(results,indent=2,allow_nan=False))
    (output_dir/'detailed_runs.json').write_text(json.dumps(scenarios,indent=2,allow_nan=False))
    with open(output_dir/'scoreboard.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
    flat=[{k:v for k,v in r.items() if isinstance(v,(int,float,str))} for runs in scenarios.values() for r in runs]
    with open(output_dir/'seed_results.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(flat[0]));w.writeheader();w.writerows(flat)
    weekly=[]
    for name,runs in scenarios.items():
        for r in runs:
            for i,snapshot in enumerate(r['snapshots']):
                row=dict(scenario=name,seed=r['seed'],week=i+1,finished=r['weekly_finished'][i],
                         stopped_hours=r['weekly_stopped'][i],stock=snapshot['stock'],
                         in_transit=snapshot['in_transit'],on_order=snapshot['on_order'])
                row.update({f'queue_WC{j+1}':v for j,v in enumerate(snapshot['queue'])});weekly.append(row)
    with open(output_dir/'weekly_results.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(weekly[0]));w.writeheader();w.writerows(weekly)
    template=ANIMATION_TEMPLATE
    (output_dir/'animation.html').write_text(template.replace('/*RESULTS_JSON*/null',json.dumps(results)))
    return scenarios,results



In [5]:
scenarios, animation_results = run_all(OUTPUT_DIR)
baseline = scenarios["baseline"]
print("\nMean [minimum, maximum] over seeds 1–10")
for name, block in animation_results["scenarios"].items():
    print(name)
    for metric, avg in block["scoreboard"].items():
        span = block["ranges"][metric]
        print(f"  {metric}: {avg:.4f} [{span['min']:.4f}, {span['max']:.4f}]")


baseline {'units_per_week': 299.9615, 'cost_per_unit_eur': np.float64(53.0308), 'kg_co2e_per_unit': np.float64(1.4327), 'hours_stopped': 6.7352}
disruption {'units_per_week': 299.9615, 'cost_per_unit_eur': np.float64(53.0287), 'kg_co2e_per_unit': np.float64(1.4327), 'hours_stopped': 15.5879}
countermeasure {'units_per_week': 299.9615, 'cost_per_unit_eur': np.float64(53.3066), 'kg_co2e_per_unit': np.float64(1.4308), 'hours_stopped': 2.8694}

Mean [minimum, maximum] over seeds 1–10
baseline
  units_per_week: 299.9615 [299.9615, 299.9615]
  cost_per_unit_eur: 53.0308 [52.9997, 53.0517]
  kg_co2e_per_unit: 1.4327 [1.4326, 1.4327]
  hours_stopped: 6.7352 [0.0000, 28.7285]
disruption
  units_per_week: 299.9615 [299.9615, 299.9615]
  cost_per_unit_eur: 53.0287 [52.9950, 53.0517]
  kg_co2e_per_unit: 1.4327 [1.4326, 1.4327]
  hours_stopped: 15.5879 [0.0000, 88.5276]
countermeasure
  units_per_week: 299.9615 [299.9615, 299.9615]
  cost_per_unit_eur: 53.3066 [53.2743, 53.3311]
  kg_co2e_per_unit:

## Unchanged official baseline check

In [6]:
# ===================== SSCM 2026 · GROUP PROJECT CHECK CELL · v1.0 =====================
# Copy this cell into your notebook UNCHANGED and run it after your baseline run.
# It needs two names defined earlier in your notebook:
#   DATA_DIR = the folder that holds your CSV files, e.g. DATA_DIR = "my_product_data"
#   baseline = your baseline results: one dictionary per seed (a list of 10), or one
#              dictionary with the means over the seeds. Keys (all numbers):
#     weeks_simulated      weeks in the run (52)
#     hours_simulated      clock hours of the run; the same clock as avg_hours_on_line
#     units_started        units that entered the line
#     units_finished       units that left the last station
#     key_components_used  key components taken from stock into the line
#     avg_units_on_line    time-average of units between start and finish
#     avg_hours_on_line    average hours from start to finish of a finished unit
#     cost_per_unit_eur    one-year total cost divided by units finished
# The grader runs the original of this cell on your files. Do not edit it.
import hashlib, math, os, datetime
import pandas as pd

CHECK_VERSION = "1.0"
REQUIRED = {
    "products.csv": ["product_id", "product_name", "frame_material", "mass_kg", "list_price_eur"],
    "bom.csv": ["product_id", "component_id", "component_name", "qty_per_unit", "mass_kg",
                "unit_cost_eur", "supplier_id", "added_at_workcenter"],
    "workcenters.csv": ["workcenter_id", "workcenter_name", "seq", "num_servers", "shift_hours",
                        "days_per_week", "availability", "process_time_cv", "fixed_cost_eur_week"],
    "routing.csv": ["product_id", "seq", "workcenter_id", "process_time_min", "setup_time_min"],
    "process_environment.csv": ["workcenter_id", "kw_running", "kw_idle", "waste_kg_per_unit",
                                "water_l_per_unit"],
    "suppliers.csv": ["supplier_id", "supplier_name", "country", "component_id", "unit_price_eur",
                      "promised_lead_days", "lead_time_mean_days", "lead_time_sd_days",
                      "capacity_per_month", "min_per_month", "kg_co2e_per_unit", "location_id"],
    "locations.csv": ["location_id", "name", "type", "country", "lat", "lon"],
    "network_lanes.csv": ["lane_id", "origin_id", "destination_id", "mode_id", "distance_km",
                          "transit_days_mean", "transit_days_sd"],
    "freight_rates.csv": ["lane_id", "mode_id", "rate_eur_per_shipment", "rate_eur_per_tkm",
                          "min_charge_eur"],
    "sources.csv": ["file", "row_id", "column", "value", "source_type", "source_or_calculation",
                    "sanity_check"],
}
MAY_BE_BLANK = {("bom.csv", "supplier_id")}           # parts not bought from a modelled supplier
MAY_BE_NEGATIVE = {("locations.csv", "lat"), ("locations.csv", "lon")}
KEYS = ["weeks_simulated", "hours_simulated", "units_started", "units_finished",
        "key_components_used", "avg_units_on_line", "avg_hours_on_line", "cost_per_unit_eur"]

def _cell_fingerprint():
    try:
        return hashlib.sha256(get_ipython().user_ns["_ih"][-1].strip().encode()).hexdigest()[:12]
    except Exception:
        return "n/a"

def _runs():
    b = globals().get("baseline")
    runs = b if isinstance(b, (list, tuple)) else [b]
    if b is None or not runs:
        raise ValueError("no `baseline` defined: run your baseline first")
    for i, r in enumerate(runs, 1):
        missing = [k for k in KEYS if k not in r]
        if missing:
            raise ValueError(f"baseline run {i} lacks {missing}")
        for k in KEYS:
            if not isinstance(r[k], (int, float)) or not math.isfinite(r[k]) or r[k] < 0:
                raise ValueError(f"baseline run {i}: {k} = {r[k]!r} is not a number ≥ 0")
        if r["units_finished"] <= 0 or r["hours_simulated"] <= 0 or r["weeks_simulated"] <= 0:
            raise ValueError(f"baseline run {i}: units_finished, hours and weeks must be > 0")
    return runs

def _verdict(name, ok, lines):
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
    for line in lines:
        print("      " + line)
    return ok

data, results = {}, {}
folder = globals().get("DATA_DIR", ".")
h = hashlib.sha256()

# ---- T1 Format ----------------------------------------------------------------------
problems = []
for fname, cols in REQUIRED.items():
    path = os.path.join(folder, fname)
    if not os.path.exists(path):
        problems.append(f"{fname}: file missing in {folder!r}")
        continue
    h.update(open(path, "rb").read())
    df = pd.read_csv(path)
    df.columns = [str(c) for c in df.columns]
    data[fname] = df
    lacking = [c for c in cols if c not in df.columns]
    if lacking:
        problems.append(f"{fname}: column(s) missing or misspelt: {lacking}")
    if len(df) == 0:
        problems.append(f"{fname}: no rows")
    for c in [c for c in cols if c in df.columns]:
        blanks = df[c].isna() | (df[c].astype(str).str.strip() == "")
        if blanks.any() and (fname, c) not in MAY_BE_BLANK:
            problems.append(f"{fname}: {c} is empty in row(s) {list(df.index[blanks] + 2)}")
        num = pd.to_numeric(df[c], errors="coerce")
        if (num < 0).any() and (fname, c) not in MAY_BE_NEGATIVE:
            problems.append(f"{fname}: {c} is negative in row(s) {list(df.index[num < 0] + 2)}")
if "workcenters.csv" in data and "availability" in data["workcenters.csv"]:
    av = pd.to_numeric(data["workcenters.csv"]["availability"], errors="coerce")
    if ((av <= 0) | (av > 1) | av.isna()).any():
        problems.append("workcenters.csv: availability must be a share between 0 and 1")
if "routing.csv" in data and "process_time_min" in data["routing.csv"]:
    if (pd.to_numeric(data["routing.csv"]["process_time_min"], errors="coerce") <= 0).any():
        problems.append("routing.csv: every process_time_min must be above 0")
results["T1"] = _verdict("T1 Format", not problems,
                         problems[:12] + (["… and more"] if len(problems) > 12 else [])
                         or ["all 10 files present, columns exact, no blanks or negatives"])

# ---- baseline numbers for T2–T5 -------------------------------------------------------
try:
    runs = _runs()
    base_error = None
except Exception as e:
    runs, base_error = [], str(e)
if base_error:
    for t in ["T2 Capacity", "T3 Little's Law", "T4 Flow balance", "T5 Margin"]:
        results[t[:2]] = _verdict(t, False, [base_error])
else:
    try:
        prod = data["products.csv"].iloc[0]
        pid = prod["product_id"]
        wc = data["workcenters.csv"].set_index("workcenter_id")
        rt = data["routing.csv"]
        rt = rt[rt["product_id"] == pid]
        minutes = rt.groupby("workcenter_id")["process_time_min"].sum()
        unknown = [w for w in minutes.index if w not in wc.index]
        avail_min_week = (wc["num_servers"] * wc["shift_hours"] * 60 * wc["days_per_week"]
                          * wc["availability"])
        # ---- T2 Capacity: load offered by the units started, per station
        lines, ok = [], not unknown and len(minutes) > 0
        if unknown:
            lines.append(f"routing.csv names stations not in workcenters.csv: {unknown}")
        worst = {}
        for r in runs:
            for w, m in minutes.items():
                if w in wc.index:
                    share = r["units_started"] * m / (avail_min_week[w] * r["weeks_simulated"])
                    worst[w] = max(worst.get(w, 0), share)
        for w, share in worst.items():
            flag = "  <- 100 % or more" if share >= 1 else ""
            lines.append(f"{w} {wc.loc[w, 'workcenter_name']}: busy {share:.1%} of available time{flag}")
            ok = ok and share < 1
        results["T2"] = _verdict("T2 Capacity", ok, lines)
        # ---- T3 Little's Law: L = λ × W, within 10 %
        lines, ok = [], True
        for i, r in enumerate(runs, 1):
            lam = r["units_finished"] / r["hours_simulated"]
            lw = lam * r["avg_hours_on_line"]
            gap = abs(r["avg_units_on_line"] - lw) / lw if lw > 0 else float("inf")
            ok = ok and gap <= 0.10
            if (gap > 0.10 and len(lines) < 3) or i == 1:
                lines.append(f"run {i}: L = {r['avg_units_on_line']:.2f}  vs  λ×W = {lam:.3f}/h × "
                             f"{r['avg_hours_on_line']:.2f} h = {lw:.2f}  (gap {gap:.1%})")
        results["T3"] = _verdict("T3 Little's Law", ok, lines + [f"{len(runs)} run(s) checked"])
        # ---- T4 Flow balance
        bom = data["bom.csv"]
        bom = bom[bom["product_id"] == pid]
        keys = bom[bom["component_id"].isin(data["suppliers.csv"]["component_id"])]
        lines, ok = [], len(keys) == 1
        if len(keys) != 1:
            lines.append(f"expected one key component (in bom.csv and suppliers.csv), found {len(keys)}")
        else:
            q = float(keys["qty_per_unit"].iloc[0])
            n_opt = int((data["suppliers.csv"]["component_id"] == keys["component_id"].iloc[0]).sum())
            if n_opt < 2:
                lines.append(f"note: only {n_opt} sourcing option in suppliers.csv (the brief asks for two)")
            for i, r in enumerate(runs, 1):
                s, f, u = r["units_started"], r["units_finished"], r["key_components_used"]
                good = f <= s + 0.5 and f * q - 0.5 <= u <= s * q + 0.5
                ok = ok and good
                if not good or i == 1:
                    lines.append(f"run {i}: started {s:,.0f}, finished {f:,.0f}, key components used "
                                 f"{u:,.0f} (allowed {f*q:,.0f} to {s*q:,.0f})")
        results["T4"] = _verdict("T4 Flow balance", ok, lines)
        # ---- T5 Margin
        price = float(prod["list_price_eur"])
        material = float((bom["qty_per_unit"] * bom["unit_cost_eur"]).sum())
        costs = [r["cost_per_unit_eur"] for r in runs]
        lines = [f"cost per unit {min(costs):,.2f} to {max(costs):,.2f} EUR vs list price {price:,.2f} EUR"]
        if min(costs) < material:
            lines.append(f"warning: below the material cost alone ({material:,.2f} EUR per unit)")
        results["T5"] = _verdict("T5 Margin", max(costs) < price, lines)
    except Exception as e:
        for t in ["T2 Capacity", "T3 Little's Law", "T4 Flow balance", "T5 Margin"]:
            if t[:2] not in results:
                results[t[:2]] = _verdict(t, False, [f"could not run: {e}"])

print("-" * 78)
print(f"{sum(results.values())} of 5 tests passed  ·  {'ALL PASS' if all(results.values()) and len(results) == 5 else 'NOT YET'}")
print(f"check cell v{CHECK_VERSION} · cell {_cell_fingerprint()} · data {h.hexdigest()[:12]} · "
      f"{datetime.datetime.now():%Y-%m-%d %H:%M}")
# ======================= end of check cell · do not edit above ==========================

PASS  T1 Format
      all 10 files present, columns exact, no blanks or negatives
PASS  T2 Capacity
      WC1 Housing and driver assembly: busy 41.7% of available time
      WC2 Electronics and wiring: busy 55.6% of available time
      WC3 Battery installation and closure: busy 41.7% of available time
      WC4 Firmware and functional test: busy 83.3% of available time
      WC5 Final inspection and packing: busy 27.8% of available time
PASS  T3 Little's Law
      run 1: L = 13.48  vs  λ×W = 7.499/h × 1.80 h = 13.48  (gap 0.0%)
      10 run(s) checked
PASS  T4 Flow balance
      run 1: started 15,600, finished 15,598, key components used 15,599 (allowed 15,598 to 15,600)
PASS  T5 Margin
      cost per unit 53.00 to 53.05 EUR vs list price 117.64 EUR
------------------------------------------------------------------------------
5 of 5 tests passed  ·  ALL PASS
check cell v1.0 · cell n/a · data 229d2c6576a7 · 2026-11-05 16:02


## Additional checks and sensitivities

In [7]:
"""Run substantive checks and ten-seed sensitivity experiments."""
import csv,json,math
from statistics import mean


def verify_and_sensitivity():
    runs=json.loads((OUTPUT_DIR/'detailed_runs.json').read_text())
    validate_runs([r for rr in runs.values() for r in rr])
    # All numeric dataset fields (except reusable transport factors) must have an
    # exactly matching source row. Catch stale values after editing a CSV.
    sources=read_csv('sources.csv')
    lookup={(r['file'],r['row_id'],r['column']):r for r in sources}
    ids={'products.csv':['product_id'],'bom.csv':['product_id','component_id'],
         'workcenters.csv':['workcenter_id'],'routing.csv':['product_id','seq'],
         'process_environment.csv':['workcenter_id'],'suppliers.csv':['supplier_id'],
         'locations.csv':['location_id'],'network_lanes.csv':['lane_id'],'freight_rates.csv':['lane_id']}
    checked=0
    for file,keys in ids.items():
        for row in read_csv(file):
            rid='|'.join(row[k] for k in keys)
            for col,val in row.items():
                try:num=float(val)
                except (ValueError,TypeError):continue
                src=lookup[file,rid,col]
                assert math.isclose(float(src['value']),num), (file,rid,col,'source mismatch')
                assert src['source_or_calculation'] and src['sanity_check']
                checked+=1
    # Matched random streams reproduce bit-for-bit for the same seed.
    a=run_scenario(1,'baseline');b=run_scenario(1,'baseline')
    assert a==b
    # Zero-duration disruption has no effect on any operating metric.
    zero=run_scenario(1,'disruption',{'disruption_duration_weeks':0})
    assert all(a[k]==zero[k] for k in SCORE_KEYS)
    # Unqualified backup cannot deliver; identical to unmitigated disruption.
    no_backup=run_scenario(1,'countermeasure',{'backup_qualified':False})
    d=run_scenario(1,'disruption')
    assert all(no_backup[k]==d[k] for k in SCORE_KEYS)
    # Force the monthly capacity limit to bind. No month may exceed allocation.
    original=SUPPLIERS['SUP-CN']['capacity_per_month']
    try:
        SUPPLIERS['SUP-CN']['capacity_per_month']='1000'
        constrained=run_scenario(1)
        validate_runs([constrained])
        assert any(o['accepted_at']>o['ordered_at'] for o in constrained['orders'] if not o['initial'])
    finally:SUPPLIERS['SUP-CN']['capacity_per_month']=original
    # Freight charged on material consumed includes the opening inventory.
    assert math.isclose(a['transport_eur']/a['key_components_used'],freight(CONFIG['order_quantity'],'SUP-CN')[0]/CONFIG['order_quantity'])
    cases={
      'wc4_plus_20pct':{'wc4_time_multiplier':1.2},
      'production_intensity_minus_20pct':{'production_intensity':64},
      'production_intensity_plus_20pct':{'production_intensity':96},
      'battery_price_minus_20pct':{'battery_price_multiplier':.8},
      'battery_price_plus_20pct':{'battery_price_multiplier':1.2},
      'cn_lead_minus_20pct':{'cn_lead_multiplier':.8},
      'cn_lead_plus_20pct':{'cn_lead_multiplier':1.2}}
    out=[]
    for label,override in cases.items():
        rr=[run_scenario(seed,'baseline',override) for seed in SEEDS];validate_runs(rr)
        for metric in SCORE_KEYS:
            v=[r[metric] for r in rr]
            out.append(dict(case=label,metric=metric,mean=mean(v),minimum=min(v),maximum=max(v)))
        print(label,round(mean(r['cost_per_unit_eur'] for r in rr),3),round(mean(r['units_per_week'] for r in rr),3))
    with open(OUTPUT_DIR/'sensitivity.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(out[0]));w.writeheader();w.writerows(out)
    summary=dict(status='PASS',scenario_runs=30,sensitivity_runs=70,numeric_source_fields_checked=checked,
        checks=['Flow balance','Inventory conservation','Weekly aggregation','Cost reconciliation',
                'Monthly aggregate supplier capacity, including forced binding case','Nonnegative queues and stock',
                'Paired deterministic seed reproduction','Zero-disruption equivalence','Unqualified backup equivalence',
                'Opening-inventory freight included in consumed cost','Exact numeric CSV/source consistency'],
        interpretation='Sensitivity lead-time changes also recalculate inventory policy; not a fixed-policy stress test.')
    (OUTPUT_DIR/'validation_report.json').write_text(json.dumps(summary,indent=2))
    return summary


print(verify_and_sensitivity())


wc4_plus_20pct 53.477 295.571
production_intensity_minus_20pct 53.003 299.962
production_intensity_plus_20pct 53.058 299.962
battery_price_minus_20pct 51.499 299.962
battery_price_plus_20pct 54.562 299.962
cn_lead_minus_20pct 53.028 299.962
cn_lead_plus_20pct 53.03 299.962
{'status': 'PASS', 'scenario_runs': 30, 'sensitivity_runs': 70, 'numeric_source_fields_checked': 141, 'checks': ['Flow balance', 'Inventory conservation', 'Weekly aggregation', 'Cost reconciliation', 'Monthly aggregate supplier capacity, including forced binding case', 'Nonnegative queues and stock', 'Paired deterministic seed reproduction', 'Zero-disruption equivalence', 'Unqualified backup equivalence', 'Opening-inventory freight included in consumed cost', 'Exact numeric CSV/source consistency'], 'interpretation': 'Sensitivity lead-time changes also recalculate inventory policy; not a fixed-policy stress test.'}


## Animation
Select a scenario and a trace seed. Scoreboard tiles always show means over ten seeds.

In [ ]:
from IPython.display import HTML, display
display(HTML((OUTPUT_DIR/"animation.html").read_text()))


# Model choices and limits

## Real product, hypothetical operations
The case represents JBL Flip 6. Product data: 0.55 kg net, 0.92 kg gross, 17.28 Wh battery, public German displayed retail price EUR 139.99 gross / EUR 117.64 net for comparison. The listing was marked sold out when checked. This is not verified factory sales revenue.
Cologne factory, BOM mass allocation, processing times, supplier compatibility, purchasing costs and lead-time distributions are modeled. PKCELL and VOLTRONIC are candidate identities, not verified JBL suppliers. The 3.6 V-equivalent, 17.28 Wh reference replaces the incompatible original 7.4 V/19.24 Wh pack. Both modeled sourcing options must provide an electrically and mechanically approved pack. The backup scenario assumes this qualification; it does not establish it in reality.

## Clock and production
52 weeks, 40 scheduled hours/week, 5 workdays/week. Four model weeks (20 workdays) form one planning month: 13 planning months/year. All model hours are working hours, not wall-clock hours. Calendar transport days are converted at 5/7; holidays excluded.
Demand releases 300 speakers/week uniformly. Five serial stations with infinite interstation buffers; no lost sales or abandonment. Queue recovery after an interruption is allowed. Ten-minute setup per 100 units at every station. Availability lengthens elapsed service, rather than simulating individual breakdown events. Gamma processing times are sampled per station and unit. This allows paired random inputs across scenarios despite changed event ordering.
WC 4 is the bottleneck: 2160 available minutes/(6+10/100)=354.10/week. At 7.2 minutes, capacity becomes 295.89/week and demand exceeds capacity. The official check excludes setups and shows 83.3% baseline processing load; including setups it is 84.7%.

## Inventory policy
CN mean = 20 production workdays + (1 origin truck +45 sea +2 destination truck +18 dwell) calendar days*5/7 =67.142857 workdays. SD retained at 11.2 workdays (16.68% CV). Draws are normal, floored at 41 workdays to accommodate a 40-workday downstream leg; actual mean is consequently slightly above the nominal untruncated mean. DE lead is normal 10 workdays, SD 2, floored at 1.
Service target assumption: z=1.645, approximately 95% one-sided normal cycle service. Safety stock=ceil(1.645*300*11.2/5)=1106. Reorder point=ceil(300*67.142857/5+1106)=5135. Q=1000. Opening on-hand=1106+500=1606; opening pipeline=round(4028.571)=4029, spread over the mean lead time. This is an approximate stationary initialization, not empirically calibrated; it avoids buying 22 weeks of physical stock solely to bridge an empty startup pipeline. The initial residual lot is part of an earlier regular purchase lot, not a new tiny shipment.
Reorder is based on on-hand plus all outstanding orders. Emergency DE orders enter that position and reduce future CN ordering. Delivery overlap and capacity queues mean the nominal 95% target is not guaranteed; baseline shortages remain possible. Do not lower stock merely to force a desired scenario outcome.

## Disruption and response
At working hour 800 (start of week 21), CN export-port access stops for 120 working hours, until start of week 24. The real analogy is Yantian 2021; the exact three-week complete block is hypothetical. A CN order's planned port departure is planned plant arrival minus 40 workdays downstream travel. Departures falling inside the window wait until it ends. Cargo already past the port is unaffected. Random lead includes all transport; lane time is not added again. The port is assumed to clear delayed departures on reopening without an additional port-capacity queue.
The countermeasure orders 900 DE packs (3*300) one workday after the disruption is detected. DE delivery lead and monthly allocation still apply. A EUR 300 annual prequalification/availability allowance (8 h*EUR 30+EUR 60 tests) is charged once. This is a precautionary fixed response, not an optimized or perfect-foresight purchasing policy. The backup is not instant, free or guaranteed compatible in reality.
Supplier monthly allocation is enforced on the aggregate accepted purchase quantity:2500 CN,1500 DE per 160 hour month. Overflow orders wait until a month has capacity; lead starts at acceptance. Limits are allocation/acceptance limits, not a detailed supplier production line. Minimum-per-month is enforced conservatively by making every new order at least that minimum. Months without an order are inactive purchasing months, without a take-or-pay obligation. Pre-year pipeline is assumed already allocated.

## Accounting and footprint
Battery lots are FIFO and retain supplier price, freight per purchased lot and embedded footprint. Expense battery and inbound freight on consumption, including beginning stock; unused ending inventory is carried as an asset. This is an accrual production-cost comparison, not cash payments in the year. Title to transit goods is assumed to pass on receipt. Holding cost is 20% of time-average landed on-hand inventory value; transit and detailed non-battery WIP excluded.
Non-battery kits are charged on finished speakers; battery costs and emissions enter at installation, including the tiny unfinished year-end battery balance. Annual fixed station costs are 52*sum(cost per server*servers). EUR 100/tCO 2 e applies to included emissions. Backup qualification is an additional fixed operating expense disclosed separately. The EUR 100/week workplace overhead allowance is assumed to cover electricity purchases as well as space/equipment; there is no extra electricity-price parameter or duplicate energy charge. Factory electricity emissions are calculated explicitly, including setups and clipped year-end work.
Battery production proxy=0.01728 kWh*80 kgCO 2 e/kWh. IVL's 61–106 range concerns vehicle NMC batteries, not a measured consumer-cell or supplier footprint. Apply the proxy to whole-battery production; remove the earlier unsupported separate assembly factors to avoid double counting. Both suppliers share the same production proxy; differences are only inbound transport. Supplier factors already include this inbound transport and it is not added twice. The model does not claim German-origin cells.
Scope is battery production proxy, inbound battery transport and factory electricity only. Other speaker materials, use, disposal, building energy and upstream fuel emissions are excluded. UBA electricity factor is directCO 2, used as a partial CO 2 e proxy; DESNZ factors are direct transport operation, not full fuel-cycle emissions. No full-product LCA claim is made.

## Outputs and interpretation
Four scoreboard fields are means and min/max across ten seeds. Annual averages can hide temporary loss and recovery, so inspect weekly output, queues, stock and starvation too. Starvation hours measure the battery station waiting with queued work and no stock; upstream stations can still produce into queues. They are not simultaneous shutdown hours of all five stations.
Snapshots occur after completions/deliveries and before next-week releases at week boundaries. Animation uses seed 1 initially and provides all ten seeds; the scoreboard always uses ten-seed means. in_transit counts dispatched but unreceived packs; on_order also includes production/port waiting. Moving symbols are schematic, not a replay of individual jobs.
Finite-horizon Little's Law reports time-average WIP including unfinished units but mean flow time for completed units. A small terminal residual is expected; the official 10% tolerance applies to baseline. Sensitivity with overloaded WC 4 can have a larger residual and is not a valid stable-baseline design.

## Verification
All five unchanged official baseline checks pass. Additional tests cover exact numeric source consistency (141 fields), stock conservation, flow balance, cost sums, weekly aggregation, bounded aggregate supplier capacity including a binding test, reproducibility, zero-length disruption and disabled backup equivalence. Seven sensitivities each run ten seeds. Lead-time sensitivity recalculates the reorder point and opening pipeline; it is not a fixed-policy lead-time shock. Browser rendering was unavailable in the execution environment; animation JSON and JavaScript were checked separately.


# Public evidence — checked 5 November 2026

## Product and battery
JBL Flip 6, German public displayed price EUR139.99 gross; listing marked sold out. Net comparison EUR117.64. https://de.jbl.com/JBLFLIP6BLKEU.html
Manufacturer datasheet: 0.55 kg net, 0.92 kg gross; Li-ion polymer battery 17.28 Wh (3.6 V/4.8 Ah equivalent); woofer, tweeter and two passive radiators. https://de.jbl.com/on/demandware.static/-/Sites-masterCatalog_Harman/default/dw0a39b813/pdfs/JBL_Flip_6_SpecSheet_English.pdf
RhineSound is the teaching case name for this real product. Cologne assembly, candidate suppliers, BOM allocations, purchasing terms and work times are hypothetical. There is no claim that JBL uses this supply chain or that either candidate supplies an approved Flip 6 pack.

## Battery price
Subtel lists a Flip 6-compatible aftermarket battery at EUR19.95 gross; it advertises a different capacity. This is a retail comparison only, not an OEM or wholesale quote. Modeled prices EUR7.50 and EUR12 remain guesstimates with explicit cost build-ups. https://www.subtel.de/akkus/bluetooth-lautsprecher-box/jbl-box-akkus/flip6

## Battery footprint
Emilsson & Dahllöf (2019), IVL report C444: vehicle NMC battery production range 61–106 kgCO2e/kWh, with chemistry, energy-mix and scale uncertainty. Model 80 is a selected proxy, not a measured consumer-cell factor. Use it for whole-battery production to avoid counting assembly twice; both suppliers share it. Only modeled inbound transport differs. https://ivl.diva-portal.org/smash/get/diva2:1549551/FULLTEXT01.pdf

## Lead time
Maersk gives a broad ocean-shipping range of 20–45 days, distinguishing port transit and door-to-door time. Model selects 45 calendar days at sea plus explicit production, trucking and dwell allowances. This does not verify the actual route or supplier delivery promise. https://www.maersk.com/de-de/support/faqs/2026/03/10/how-long-does-ocean-freight-take

## Real disruption analogy
Yantian terminal operator's 23 June 2021 notice announces restoration of full operations from 24 June after COVID restrictions and gradual recovery. It supports port-access disruption as the event type. Three weeks of complete modeled export access loss is a scenario assumption, not a claim that the historical port was fully closed for exactly three weeks. https://www.yict.com.cn/article/detail/8948.html?locale=en_US

## Course requirements
https://th-koln-bartnik.github.io/sscm-2026-student/project.html
Animation schema and original template: https://github.com/TH-Koln-Bartnik/sscm-2026-student/tree/main/week04/project

Existing DESNZ transport and UBA electricity references retained from submitted dataset; not independently re-downloaded in this revision. AI output is not used as a data source.


# Updated literature connection and draft AI note

## Whitney et al. (three sentences)
Whitney, Luo and Heller compare the Aisin Seiki fire and Riken earthquake recoveries to ask when temporary sourcing diversification is feasible. They find that product and production specificity can prevent an alternative source from substituting, so an additional supplier must have suitable capability and capacity. In this model, the explicitly prequalified DE option reduces mean disruption shortage time from 15.59 to 2.87 hours, while disabling qualification removes that benefit; the result is conditional on qualification, a one-day activation and the modeled 1500-pack monthly allocation.

Whitney, D. E., Luo, J., & Heller, D. A. (2014). The benefits and constraints of temporary sourcing diversification in supply chain disruption and recovery. *Journal of Purchasing and Supply Management, 20*(4), 238–250. https://doi.org/10.1016/j.pursup.2014.06.001

## Ivanov (three sentences)
Ivanov examines how network measures and dynamic performance indicators complement one another in assessing supply-chain and sourcing resilience. Structural connectivity alone does not establish whether a sourcing option improves performance over time. Our extra DE link leaves annual output essentially unchanged but reduces mean temporary shortage time by 12.72 hours at higher cost, demonstrating why the weekly path and four-metric scoreboard are needed.

Ivanov, D. (2025). What network and performance indicators can tell us about supply chain and sourcing resilience (and what they cannot). *Journal of Purchasing and Supply Management, 31*(3), 101014. https://doi.org/10.1016/j.pursup.2025.101014

Bibliographic details and research summaries retained from the team's submitted literature document and checkpoint feedback. No new claim of full-text re-verification is made in this revision.

## Draft AI note — adapt only after your own review
AI assisted with auditing the submitted dataset and feedback, implementing the three-scenario model, locating public reference data, revising provenance, and preparing results and explanatory text. Automated checks executed the unchanged course check cell, stock/flow/cost reconciliation, numeric source consistency, matched-seed comparisons, capacity constraints and sensitivity experiments. An earlier model mistake was that supplier capacity was checked per order rather than across orders in a month; the revision uses a monthly capacity ledger and a test that forces it to bind. Public evidence is linked in sources.csv; unverified prices, lead times, compatibility and model policy remain marked as assumptions rather than AI-sourced facts.

Team verification still to record truthfully: run all cells in Colab, inspect animation in your browser, and confirm that every member can explain the assumptions and code. Do not claim these team actions occurred until you perform them.
